## Middleware

In [ ]:
import os
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver # save in HDD
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [ ]:
model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google_genai"
)

agent=create_agent(
    model=model,checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model=model,
        trigger=("messages",10) # when message size more then 10 trigger summrization
        ,keep=("messages",4)) # keep last 4 message
    ]
)

In [3]:
config = { "configurable" : {"thread_id":"test-1"} }
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

In [5]:
for q in questions:
    response = agent.invoke({"messages":HumanMessage(content=q)},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user\'s primary goal is to perform basic arithmetic calculations by asking questions and receiving answers.\n\n## SUMMARY\n\nThe conversation has consisted of the user posing several basic arithmetic problems (addition, multiplication, division), and the AI successfully providing the correct numerical answers for each.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nThe next step is to answer the user\'s last arithmetic question: "What is 15-7?".', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='b7a11aa6-696b-483c-9a18-18029f65bf9b'), AIMessage(content='15 - 7 = 8', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05ccb-0571-7dd1-9f92-c45215bcfb66-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65

#### Token

In [18]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage,SystemMessage
from langchain.tools import tool
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [19]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


In [20]:
model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
)

In [21]:
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",500),
            keep=("tokens",200),
        ),
    ]
)

In [22]:
config = {"configurable":{"thread_id":"user-1"}}
# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [26]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":HumanMessage(content=f"Find hotels in {city}")},
        config=config
        )
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~372 tokens, 5 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT  \nThe user wants to find hotels in Singapore.\n\n## SUMMARY  \n- The assistant used the `search_hotels` tool with the query `"city": "Singapore"`.  \n- The tool returned a list of three hotels:  \n  1. Grand Hotel – 5★, $350/night, spa, pool, gym  \n  2. City Inn – 4★, $180/night, business center  \n  3. Budget Stay – 3★, $75/night, free Wi‑Fi  \n- The assistant presented the results in a table and asked the user what to do next (room details, filtering, different city, etc.).\n\n## ARTIFACTS  \nNone.\n\n## NEXT STEPS  \nAwait the user’s next request (e.g., filtering, room details, availability, or a different city).', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='5451807a-fd2f-4f5d-821b-dab1998fcb8f'), HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='f4b85887-ca55-4f65-9169-e8d99fad074a'), A

### Fraction Through

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

### Human In the loop Middleware

##### Pause agent execution for human approval, editing, or rejection of tool calls before they execute

##### Human-in-the-loop is useful for the following:

###### High-stakes operations requiring human approval (e.g. database writes, financial transactions).
###### Compliance workflows where human oversight is mandatory.
###### Long-running conversations where human feedback guides the agent.

In [26]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv
load_dotenv()


True

In [25]:
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [27]:
model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
)

In [43]:
agent=create_agent(
    model=model,
    tools = [send_email_tool,read_email_tool],
    checkpointer=InMemorySaver(),
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
            
        )
    ]
)

In [44]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke({"messages":[HumanMessage("send the mail to the ravi@gmail.com with subject Hi body How Are You ?")]},config)


In [34]:
result

{'messages': [HumanMessage(content='send the mail to the ravi@gmail.com with subject Hi body How Are You ?', additional_kwargs={}, response_metadata={}, id='3b8edeb6-06eb-405b-bc8b-3b383a9fb12d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the send_email_tool. Provide body, recipient, subject.', 'tool_calls': [{'id': 'fc_1c902c76-2928-450b-86ef-c549a1f9ead3', 'function': {'arguments': '{"body":"How Are You ?","recipient":"ravi@gmail.com","subject":"Hi"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 172, 'total_tokens': 227, 'completion_time': 0.06487468, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.063580844, 'prompt_tokens_details': None, 'queue_time': 0.167330404, 'total_time': 0.128455524}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': No

In [36]:
from langgraph.types import Command
if "__interrupt__" in result:
     print("⏸️ Paused! Approving...")
     result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Email sent to **ravi@gmail.com** with subject **“Hi”** and body **“How Are You ?”**. If you need anything else, just let me know!


#### Edit

In [45]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [46]:
result

{'messages': [HumanMessage(content='send the mail to the ravi@gmail.com with subject Hi body How Are You ?', additional_kwargs={}, response_metadata={}, id='2aa3d6db-f3c2-446b-839d-6a385e0065d8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to ravi@gmail.com with subject "Hi" and body "How Are You ?" Using the send_email_tool. Let\'s do that.', 'tool_calls': [{'id': 'fc_d56e60c9-080a-4c85-8e09-cf561153b3e6', 'function': {'arguments': '{"body":"How Are You ?","recipient":"ravi@gmail.com","subject":"Hi"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 172, 'total_tokens': 244, 'completion_time': 0.107597285, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.00972057, 'prompt_tokens_details': None, 'queue_time': 0.274762292, 'total_time': 0.117317855}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service